# PADAssembler visual check — all four detectors, unified pathway

`notebooks/detector_assembly_crosscheck.ipynb` (2026-09-16) compared our assembly
against Resonet's independent CrystFEL-based assembly, but at the time Jungfrau 4M
was **not** assembled by Reborn at all — it was a bare `_to_2d(raw)` passthrough of
the pre-assembled LCLS canvas, while AGIPD/ePix10k/Eiger4M already went through
`PADAssembler`. That crosscheck notebook's `assemble_ours()` still special-cases
Jungfrau with `_to_2d`, so it never exercised the real pathway for that detector.

Since then (plan `2026-09-24-jungfrau-pad-assembler`, merged via PR #37), Jungfrau 4M
was switched onto the same `get_geometry()` → `get_assembler()` →
`src.preprocessing.pipeline.assemble_only()` pathway as the other three detectors,
using `jungfrau_8_pad_geometry_list()` (8 panels, 103mm) with real panel placement
instead of the raw canvas's arbitrary layout. That switch was verified structurally
(shape, mask coverage, bit-exactness of the *pipeline*) but — per kanban item
"Visually verify fixed panel layouts (reopened — prior verdict unconfirmed)" —
never re-confirmed visually.

**This notebook's job:** assemble all four detectors through the exact current
production pathway (`assemble_only()` + `valid_pixel_mask()`, the same functions the
training pipeline calls) and render them so panel placement, orientation, and gap
pattern can be eyeballed directly. Jungfrau is the detector of interest; the other
three are included as a sanity baseline (their pathway hasn't changed since the prior
crosscheck).

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# hdf5plugin MUST be imported before h5py to register the bitshuffle filter.
import hdf5plugin  # noqa: F401
import h5py

from src.preprocessing.geometry import get_assembler, get_geometry
from src.preprocessing.pipeline import assemble_only, valid_pixel_mask
from src.preprocessing.io import read_detector_description, read_frame

DATA_ROOT = "/data/bioxfel/user/gihan/Resonet/production"
LABEL_KEY = "entry_1/data_1/data"

PRODUCTION_FILES = {
    "AGIPD": os.path.join(DATA_ROOT, "agipd_20k", "compressed0.cxi"),
    "JUNGFRAU_4M": os.path.join(DATA_ROOT, "jungfrau_20k", "compressed0.cxi"),
    "ePix10k": os.path.join(DATA_ROOT, "epix10k_20k", "compressed0.cxi"),
    "Eiger4M": os.path.join(DATA_ROOT, "eiger4m_20k", "compressed0.cxi"),
}
for label, path in PRODUCTION_FILES.items():
    print(f"{label:12s} exists={os.path.exists(path)}  {path}")

## 1. Production assembly path

This is the *actual* training pathway, not a reimplementation: `assemble_only()` and
`valid_pixel_mask()` are imported directly from `src.preprocessing.pipeline` (the same
module `AsymmetricCXIDataset` and the eval benchmark call). No copy, no second
implementation to drift out of sync.

In [ ]:
_pair_cache = {}  # (detector, frame_idx) -> (assembled, mask, desc)
_nframes_cache = {}


def n_frames(name):
    if name not in _nframes_cache:
        with h5py.File(PRODUCTION_FILES[name], "r") as f:
            _nframes_cache[name] = int(f[LABEL_KEY].shape[0])
    return _nframes_cache[name]


def load_assembled(name, idx):
    key = (name, idx)
    if key in _pair_cache:
        return _pair_cache[key]

    cxi = PRODUCTION_FILES[name]
    raw = np.asarray(read_frame(cxi, idx), dtype=np.float32)
    desc = read_detector_description(cxi)

    pads = get_geometry(desc)
    assembler = get_assembler(desc)
    assembled = assemble_only(raw, pads, desc, assembler=assembler)
    mask = valid_pixel_mask(desc)

    meta = {
        "desc": desc,
        "raw_shape": raw.shape,
        "n_panels": len(pads),
        "assembled_shape": assembled.shape,
        "valid_frac": float(mask.mean()),
    }
    result = (assembled, mask, meta)
    _pair_cache[key] = result
    return result

## 2. Interactive viewer

Gaps/padding (`mask == False`) are tinted cyan on top of the intensity image so panel
boundaries are visible even on a sparse/low-signal frame. A scrambled, rotated, or
mirrored panel layout shows up as a gap pattern that doesn't form a sensible detector
shape (e.g. panels not aligned into rows/quadrants).

In [ ]:
w_detector = widgets.Dropdown(
    options=list(PRODUCTION_FILES.keys()), value="JUNGFRAU_4M", description="detector:"
)
w_frame = widgets.IntSlider(value=0, min=0, max=0, step=1, description="frame:")
w_out = widgets.Output()


def _refresh_frame_bounds(*_):
    w_frame.max = max(0, n_frames(w_detector.value) - 1)


def _render(*_):
    with w_out:
        w_out.clear_output(wait=True)
        assembled, mask, meta = load_assembled(w_detector.value, w_frame.value)

        pos = assembled[assembled > 0]
        vmin = float(np.percentile(pos, 1)) if len(pos) else 0.0
        vmax = float(np.percentile(pos, 99.5)) if len(pos) else 1.0

        fig, ax = plt.subplots(figsize=(7, 7), dpi=110)
        ax.imshow(assembled, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper")
        # Cyan overlay on invalid (gap/padding) pixels, alpha-blended.
        overlay = np.zeros((*mask.shape, 4))
        overlay[~mask] = (0.0, 1.0, 1.0, 0.35)
        ax.imshow(overlay, origin="upper")
        ax.set_title(
            f"{meta['desc']} — frame {w_frame.value}\n"
            f"raw={meta['raw_shape']}  panels={meta['n_panels']}  "
            f"assembled={meta['assembled_shape']}  valid_frac={meta['valid_frac']:.3f}",
            fontsize=9,
        )
        ax.axis("off")
        plt.show()


w_detector.observe(lambda c: (_refresh_frame_bounds(), _render()), names="value")
w_frame.observe(_render, names="value")
_refresh_frame_bounds()
display(widgets.VBox([widgets.HBox([w_detector, w_frame]), w_out]))
_render()

## 3. Static summary gallery (frame 0, all four detectors)

Fixed, no widgets, so it survives a headless `nbconvert --execute` run and can be
diffed/archived.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 11), dpi=130)
for ax, name in zip(axes.ravel(), PRODUCTION_FILES.keys()):
    assembled, mask, meta = load_assembled(name, 0)
    pos = assembled[assembled > 0]
    vmin = float(np.percentile(pos, 1)) if len(pos) else 0.0
    vmax = float(np.percentile(pos, 99.5)) if len(pos) else 1.0

    ax.imshow(assembled, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper")
    overlay = np.zeros((*mask.shape, 4))
    overlay[~mask] = (0.0, 1.0, 1.0, 0.35)
    ax.imshow(overlay, origin="upper")
    ax.set_title(
        f"{meta['desc']}\n{meta['assembled_shape']}, {meta['n_panels']} panels, "
        f"valid={meta['valid_frac']:.2f}",
        fontsize=10,
    )
    ax.axis("off")
fig.suptitle("Unified PADAssembler pathway — frame 0, all four detectors", fontsize=13)
fig.tight_layout()

out_dir = os.path.join(PROJECT_ROOT, "docs", "figures", "assembly_crosscheck")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "pad_assembler_unified_frame0.png")
fig.savefig(out_path, bbox_inches="tight")
print(f"saved → {out_path}")
plt.show()

## 4. Side-by-side with Resonet's independent CrystFEL assembly

Same purpose as `notebooks/detector_assembly_crosscheck.ipynb`: Resonet's
`view_cxi.py` assembles the same CXI files through a completely independent
path — no Reborn, no `PADGeometry`, no `PADAssembler` — by regex-parsing the
CrystFEL `.geom` text and scattering pixels with plain affine math. Since the
two codebases share no assembly code, agreement between them is real evidence,
not a shared bug.

`parse_geom_panels()` and `assemble_crystfel()` below are copied **verbatim**
from `../Resonet/view_cxi.py:16-94` (`assemble` renamed only) and from the
crosscheck notebook's own copy — unmodified, so the comparison is against
Resonet's real behaviour. `view_cxi.py` itself cannot be imported: it runs a
CLI and forces the TkAgg backend at import time.

Geom-file mapping (confirmed 2026-09-16, see crosscheck notebook §"Geom file
mapping"): AGIPD needs the flattened `AGIPD-2.geom` (not `AGIPD.geom`, which
addresses per-module and would make the CrystFEL 2D slicer read the wrong
pixels), so its raw frame is reshaped `(16, 512, 128) -> (8192, 128)` first.
The other three use their natively-2D raw shape directly.

Panel-number labels (checkbox below) use the same centroid formula as
`view_cxi.py:142-149` on the CrystFEL pane, and `PADAssembler.flat_indices`
panel centroids (`.geom[k].name`) on our pane — independent labellings, so a
panel carrying the same name in both panes is what confirms the two
assemblies agree on which sensor is which.

No display rotation is applied here for Jungfrau — the crosscheck notebook's
90°-CCW display correction (AGIPD/ePix10k/Eiger4M axis-convention mismatch
between `PADAssembler` and `assemble_crystfel`) was derived by correlating
against CrystFEL for those three specific detectors; Jungfrau was never in
that set since it had no `PADAssembler` output to correlate against. Applying
an unverified rotation here would defeat the point of this check, so Jungfrau
renders as-is.

In [ ]:
import re
import matplotlib.patheffects as pe

GEOM_ROOT = os.path.join(PROJECT_ROOT, "..", "Resonet", "geoms")

# name -> (CrystFEL geom file, reshape applied before the CrystFEL 2D slicer)
CRYSTFEL_GEOM = {
    "AGIPD":       ("AGIPD-2.geom", (8192, 128)),
    "JUNGFRAU_4M": ("Junfrau.geom", None),
    "ePix10k":     ("Epix10k.geom", None),
    "Eiger4M":     ("Eigar.geom",   None),
}
for _name, (_geom, _) in CRYSTFEL_GEOM.items():
    _p = os.path.join(GEOM_ROOT, _geom)
    assert os.path.exists(_p), f"missing geom for {_name}: {_p}"

_OUTLINE = [pe.withStroke(linewidth=2.2, foreground="black")]


# ---------------------------------------------------------------------------
# Verbatim from /data/bioxfel/user/gihan/Resonet/view_cxi.py:16-94 (also
# copied into notebooks/detector_assembly_crosscheck.ipynb). Copied, not
# imported: view_cxi.py executes its CLI at import and sets TkAgg. Do not
# "improve" these -- the point is to compare against Resonet as it runs.
# ---------------------------------------------------------------------------
def parse_geom_panels(geom_path):
    """Extract panel geometry from a CrystFEL .geom file.

    Returns list of dicts with: name, corner_x, corner_y, fs_x, fs_y, ss_x, ss_y,
    min_fs, max_fs, min_ss, max_ss.
    """
    panels = {}
    with open(geom_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(';') or '=' not in line:
                continue
            key, _, val = line.partition('=')
            key, val = key.strip(), val.strip()
            m = re.match(r'^(\w+)/(corner_x|corner_y|min_fs|max_fs|min_ss|max_ss|fs|ss)$', key)
            if not m:
                continue
            name, field = m.group(1), m.group(2)
            panels.setdefault(name, {})['name'] = name
            if field in ('fs', 'ss'):
                def _coeff(m):
                    if m is None: return 0.0
                    s = m.group(1)
                    return 1.0 if s in ('', '+') else (-1.0 if s == '-' else float(s))
                mx = re.search(r'([+-]?(?:[0-9.]+(?:[eE][+-]?[0-9]+)?)?)x', val)
                my = re.search(r'([+-]?(?:[0-9.]+(?:[eE][+-]?[0-9]+)?)?)y', val)
                panels.setdefault(name, {})[field + '_x'] = _coeff(mx)
                panels.setdefault(name, {})[field + '_y'] = _coeff(my)
            else:
                panels.setdefault(name, {})[field] = float(val)

    required = {'corner_x', 'corner_y', 'fs_x', 'fs_y', 'ss_x', 'ss_y',
                'min_fs', 'max_fs', 'min_ss', 'max_ss'}
    return [p for p in panels.values() if required.issubset(p)]


def assemble_crystfel(raw_frame, panels):
    """Assemble panels onto a canvas using full affine placement.

    For each pixel (i_fs, i_ss) in a panel:
        x =   corner_x + i_fs * fs_x + i_ss * ss_x
        y = -(corner_y + i_fs * fs_y + i_ss * ss_y)  # negated: CrystFEL +y=up, canvas row 0=top

    Handles any axis orientation including flipped/rotated panels.
    Returns (canvas, x_min, y_min) where x_min/y_min are the canvas origin offsets.
    """
    all_xs, all_ys = [], []
    for p in panels:
        nf = int(p['max_fs'] - p['min_fs'] + 1)
        ns = int(p['max_ss'] - p['min_ss'] + 1)
        for i_fs, i_ss in [(0, 0), (nf - 1, 0), (0, ns - 1), (nf - 1, ns - 1)]:
            all_xs.append(p['corner_x'] + i_fs * p['fs_x'] + i_ss * p['ss_x'])
            all_ys.append(-(p['corner_y'] + i_fs * p['fs_y'] + i_ss * p['ss_y']))

    x_min = int(np.floor(min(all_xs)))
    y_min = int(np.floor(min(all_ys)))
    x_max = int(np.ceil(max(all_xs))) + 1
    y_max = int(np.ceil(max(all_ys))) + 1

    canvas = np.zeros((y_max - y_min, x_max - x_min), dtype=np.float32)

    for p in panels:
        nf = int(p['max_fs'] - p['min_fs'] + 1)
        ns = int(p['max_ss'] - p['min_ss'] + 1)
        ss0 = int(p['min_ss'])
        fs0 = int(p['min_fs'])
        patch = raw_frame[ss0:ss0 + ns, fs0:fs0 + nf]

        i_fs, i_ss = np.meshgrid(np.arange(nf), np.arange(ns))
        col = np.round(p['corner_x'] + i_fs * p['fs_x'] + i_ss * p['ss_x']).astype(int) - x_min
        row = np.round(-(p['corner_y'] + i_fs * p['fs_y'] + i_ss * p['ss_y'])).astype(int) - y_min

        mask = (row >= 0) & (row < canvas.shape[0]) & (col >= 0) & (col < canvas.shape[1])
        canvas[row[mask], col[mask]] = patch[mask]

    return canvas, x_min, y_min


def label_panels(ax, panels, x_min, y_min):
    """Panel-name overlay on the CrystFEL pane, same centroid formula as view_cxi.py:142-149."""
    for p in panels:
        nf = int(p['max_fs'] - p['min_fs'] + 1)
        ns = int(p['max_ss'] - p['min_ss'] + 1)
        cx = (p['corner_x'] + (nf - 1) / 2 * p['fs_x'] + (ns - 1) / 2 * p['ss_x']) - x_min
        ry = (-(p['corner_y'] + (nf - 1) / 2 * p['fs_y'] + (ns - 1) / 2 * p['ss_y'])) - y_min
        ax.text(cx, ry, p.get('name', '?'), fontsize=9, color='cyan',
                ha='center', va='center', clip_on=True, path_effects=_OUTLINE)


def panel_centroids_ours(desc):
    """Panel -> (name, row, col) centroid on OUR assembled canvas.

    PADAssembler.flat_indices maps every pixel of the concatenated panel data to a
    flat index on the assembled canvas, in the same panel order as pads. Slicing it
    per panel and averaging gives where that panel actually landed.
    """
    asm = get_assembler(desc)
    n_cols = int(asm.shape[1])
    flat = np.asarray(asm.flat_indices)
    offsets = np.concatenate([[0], np.cumsum([int(pad.n_pixels) for pad in asm.geom])])
    out = []
    for k in range(len(asm.geom)):
        idx = flat[offsets[k]:offsets[k + 1]]
        name = asm.geom[k].name or str(k)
        out.append((name, float((idx // n_cols).mean()), float((idx % n_cols).mean())))
    return out


def label_panels_ours(ax, desc):
    """Label panels on our pane by their .geom name -- same identifier space as
    label_panels() on the CrystFEL pane."""
    for name, row, col in panel_centroids_ours(desc):
        ax.text(col, row, name, fontsize=9, color="cyan", ha="center", va="center",
                clip_on=True, path_effects=_OUTLINE)


_crystfel_pair_cache = {}
_crystfel_geom_cache = {}


def load_pair_crystfel(name, idx):
    """Assemble frame `idx` of detector `name` both ways. Returns (ours, theirs, meta)."""
    key = (name, idx)
    if key in _crystfel_pair_cache:
        return _crystfel_pair_cache[key]

    ours, mask, meta_ours = load_assembled(name, idx)

    cxi = PRODUCTION_FILES[name]
    raw = np.asarray(read_frame(cxi, idx), dtype=np.float32)
    geom_name, reshape = CRYSTFEL_GEOM[name]
    geom_path = os.path.join(GEOM_ROOT, geom_name)

    raw_cf = raw.reshape(reshape) if reshape is not None else raw
    if geom_name not in _crystfel_geom_cache:
        _crystfel_geom_cache[geom_name] = parse_geom_panels(geom_path)
    panels = _crystfel_geom_cache[geom_name]

    theirs, x_min, y_min = assemble_crystfel(raw_cf, panels)

    meta = {
        **meta_ours,
        "geom": geom_name,
        "crystfel_panels": len(panels),
        "crystfel_canvas": theirs.shape,
        "crystfel_origin": (x_min, y_min),
    }
    result = (ours, theirs, panels, meta)
    _crystfel_pair_cache[key] = result
    return result


print("Resonet CrystFEL comparison setup ready:", list(CRYSTFEL_GEOM))

In [ ]:
cf_w_detector = widgets.Dropdown(
    options=list(PRODUCTION_FILES.keys()), value="JUNGFRAU_4M", description="detector:"
)
cf_w_frame = widgets.IntSlider(value=0, min=0, max=0, step=1, description="frame:")
cf_w_labels = widgets.Checkbox(value=True, description="PAD numbers", indent=False)
cf_w_out = widgets.Output()


def _cf_refresh_frame_bounds(*_):
    cf_w_frame.max = max(0, n_frames(cf_w_detector.value) - 1)


def _cf_render(*_):
    with cf_w_out:
        cf_w_out.clear_output(wait=True)
        ours, theirs, panels, meta = load_pair_crystfel(cf_w_detector.value, cf_w_frame.value)

        fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=110)
        for ax, img, title in (
            (axes[0], ours, f"ours — Reborn PADAssembler\n{meta['assembled_shape']}  panels={meta['n_panels']}"),
            (axes[1], theirs, f"theirs — CrystFEL ({meta['geom']})\n{meta['crystfel_canvas']}  panels={meta['crystfel_panels']}"),
        ):
            pos = img[img > 0]
            vmin, vmax = (np.percentile(pos, [1, 99.5]) if len(pos) else (0.0, 1.0))
            ax.imshow(img, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper", interpolation="nearest")
            ax.set_title(title, fontsize=10)
            ax.axis("off")

        if cf_w_labels.value:
            label_panels_ours(axes[0], meta["desc"])
            x_min, y_min = meta["crystfel_origin"]
            label_panels(axes[1], panels, x_min, y_min)

        same = meta["assembled_shape"] == meta["crystfel_canvas"]
        verdict = "SAME SHAPE" if same else (
            f"DIFFERENT SHAPE ours={meta['assembled_shape']} crystfel={meta['crystfel_canvas']}"
        )
        fig.suptitle(f"{meta['desc']} — frame {cf_w_frame.value} — {verdict}", fontsize=13)
        fig.tight_layout()
        plt.show()


cf_w_detector.observe(lambda c: (_cf_refresh_frame_bounds(), _cf_render()), names="value")
cf_w_frame.observe(_cf_render, names="value")
cf_w_labels.observe(_cf_render, names="value")
_cf_refresh_frame_bounds()
display(widgets.VBox([widgets.HBox([cf_w_detector, cf_w_frame, cf_w_labels]), cf_w_out]))
_cf_render()